In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import os

# Load the clean dataset
df = pd.read_csv('../extracted/ciciot_clean.csv')
print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Loaded: (21005152, 41)
Columns: ['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label', 'Category']


In [9]:
# === STEP 1: Create binary and multiclass targets ===
df['target_binary'] = (df['Label'] != 'BENIGN').astype(int)

# Encode multiclass label
le = LabelEncoder()
df['target_multi'] = le.fit_transform(df['Label'])

# Save label mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("=== BINARY TARGET ===")
print(df['target_binary'].value_counts())
print(f"\n0 = Benign, 1 = Attack")

print("\n=== MULTICLASS ENCODING ===")
for k, v in sorted(label_mapping.items()):
    print(f"  {v:2d} → {k}")

=== BINARY TARGET ===
target_binary
1    19957844
0     1047308
Name: count, dtype: int64

0 = Benign, 1 = Attack

=== MULTICLASS ENCODING ===
   0 → BACKDOOR_MALWARE
   1 → BENIGN
   2 → BROWSERHIJACKING
   3 → COMMANDINJECTION
   4 → DDOS-ACK_FRAGMENTATION
   5 → DDOS-HTTP_FLOOD
   6 → DDOS-ICMP_FLOOD
   7 → DDOS-ICMP_FRAGMENTATION
   8 → DDOS-PSHACK_FLOOD
   9 → DDOS-RSTFINFLOOD
  10 → DDOS-SLOWLORIS
  11 → DDOS-SYNONYMOUSIP_FLOOD
  12 → DDOS-SYN_FLOOD
  13 → DDOS-TCP_FLOOD
  14 → DDOS-UDP_FLOOD
  15 → DDOS-UDP_FRAGMENTATION
  16 → DICTIONARYBRUTEFORCE
  17 → DNS_SPOOFING
  18 → DOS-HTTP_FLOOD
  19 → DOS-SYN_FLOOD
  20 → DOS-TCP_FLOOD
  21 → DOS-UDP_FLOOD
  22 → MIRAI-GREETH_FLOOD
  23 → MIRAI-GREIP_FLOOD
  24 → MIRAI-UDPPLAIN
  25 → MITM-ARPSPOOFING
  26 → RECON-HOSTDISCOVERY
  27 → RECON-OSSCAN
  28 → RECON-PINGSWEEP
  29 → RECON-PORTSCAN
  30 → SQLINJECTION
  31 → UPLOADING_ATTACK
  32 → VULNERABILITYSCAN
  33 → XSS


In [15]:
# === STEP 2: Train / Validation / Test Split (70/10/20) ===
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns 
                if c not in ['Label', 'Category', 'target_binary', 'target_multi']]

X = df[feature_cols]
y_binary = df['target_binary']
y_multi  = df['target_multi']

# First split: 80% train+val / 20% test
X_temp, X_test, y_temp_b, y_test_b, y_temp_m, y_test_m = train_test_split(
    X, y_binary, y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

# Second split: 70% train / 10% val from the 80%
X_train, X_val, y_train_b, y_val_b, y_train_m, y_val_m = train_test_split(
    X_temp, y_temp_b, y_temp_m,
    test_size=0.125,  # 0.125 x 80% = 10% of total
    random_state=42,
    stratify=y_temp_b
)

print(f"Train: {X_train.shape} — {y_train_b.value_counts().to_dict()}")
print(f"Val:   {X_val.shape}   — {y_val_b.value_counts().to_dict()}")
print(f"Test:  {X_test.shape}  — {y_test_b.value_counts().to_dict()}")

print(f"\nTrain: {len(X_train)/len(X)*100:.1f}%")
print(f"Val:   {len(X_val)/len(X)*100:.1f}%")
print(f"Test:  {len(X_test)/len(X)*100:.1f}%") 

Train: (14703605, 39) — {1: 13970490, 0: 733115}
Val:   (2100516, 39)   — {1: 1995785, 0: 104731}
Test:  (4201031, 39)  — {1: 3991569, 0: 209462}

Train: 70.0%
Val:   10.0%
Test:  20.0%


In [16]:
# === STEP 3: Scale features (fit on train only) ===
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val),   columns=feature_cols)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=feature_cols)

print(f"Train scaled: {X_train_scaled.shape}")
print(f"Val scaled:   {X_val_scaled.shape}")
print(f"Test scaled:  {X_test_scaled.shape}")
print(f"\nFeature range: [{X_train_scaled.min().min():.2f}, {X_train_scaled.max().max():.2f}]")

Train scaled: (14703605, 39)
Val scaled:   (2100516, 39)
Test scaled:  (4201031, 39)

Feature range: [0.00, 1.00]


In [17]:
# === STEP 4: Save all splits ===
import pickle, json

X_train_scaled.to_csv('../extracted/X_train.csv', index=False)
X_val_scaled.to_csv('../extracted/X_val.csv', index=False)
X_test_scaled.to_csv('../extracted/X_test.csv', index=False)
y_train_b.to_csv('../extracted/y_train_binary.csv', index=False)
y_val_b.to_csv('../extracted/y_val_binary.csv', index=False)
y_test_b.to_csv('../extracted/y_test_binary.csv', index=False)
y_train_m.to_csv('../extracted/y_train_multi.csv', index=False)
y_val_m.to_csv('../extracted/y_val_multi.csv', index=False)
y_test_m.to_csv('../extracted/y_test_multi.csv', index=False)

pickle.dump(scaler, open('../extracted/scaler.pkl', 'wb'))
pickle.dump(le, open('../extracted/label_encoder.pkl', 'wb'))

label_mapping_json = {k: int(v) for k, v in label_mapping.items()}
with open('../extracted/label_mapping.json', 'w') as f:
    json.dump(label_mapping_json, f, indent=2)

print("=== SAVED ===")
print("  X_train.csv, X_val.csv, X_test.csv")
print("  y_train/val/test binary + multi")
print("  scaler.pkl, label_encoder.pkl")
print("  label_mapping.json")
print(f"\nFinal split sizes:")
print(f"  Train: {X_train_scaled.shape}")
print(f"  Val:   {X_val_scaled.shape}")
print(f"  Test:  {X_test_scaled.shape}")

=== SAVED ===
  X_train.csv, X_val.csv, X_test.csv
  y_train/val/test binary + multi
  scaler.pkl, label_encoder.pkl
  label_mapping.json

Final split sizes:
  Train: (14703605, 39)
  Val:   (2100516, 39)
  Test:  (4201031, 39)
